# V12 – Testen & Debuggen: Prüfen, bevor du vertraust – Vertiefung zum Training

**Wo passt das Notebook hin?** Erst die Python-Werkstatt in der Lektion, dann das Training zu Hause (Browser, ca. 60 Min). Dieses Notebook ist die **Vertiefung**: Du übst dieselben Konzepte (`assert`, Testfälle als Liste von Tupeln, Bereiche und Grenzwerte, Tests gegen fehlerhafte Varianten, Traceback lesen, Fehler mit `print` und Debugger finden) an ingenieurnahen Aufgaben: Lagertemperatur, Briefporto, Bremsmessung, Schutzfunktion. Es nutzt nur Python aus V01 bis V12.

**Aufbau:** drei Stufen: *Aufwärmen* (Vorhersagen und Lesen), *Anwenden* (Tests schreiben, Fehler finden), *Knobeln* (eigene Mutanten, Wahrheitstabellen). Am Ende ein Ausblick auf `pytest`. **Es gibt keine Lösungen.** Jede Aufgabe hat einen Selbsttest: Du siehst sofort, ob du richtig liegst. Wenn nicht: Fehlermeldung lesen, nochmal versuchen, nicht gleich nachschauen.

**Arbeitsweise:** Erst die *Vorhersage* in den Kommentar schreiben, dann ausführen. Immer erst das Erwartete von Hand ausrechnen, dann den Code ansehen.

**Debugger ausprobieren:** In Codespaces (VS Code) kannst du in jeder Zelle einen Haltepunkt setzen (Klick links neben die Zeilennummer, roter Punkt) und die Zelle über das Menü neben dem Ausführen-Knopf „Zelle debuggen“ starten. Probiere das in Aufgabe 2.3.

In [ ]:
# Werkzeug-Zelle: einmal ausführen, nicht verändern.
# Sie definiert die Selbsttests und die Funktion, mit der deine Tests gegen fehlerhafte Varianten laufen.
import math
import traceback


def pruefe(ist, soll, text):
    """Vergleicht ist mit soll (exakt) und meldet das Ergebnis."""
    if ist == soll:
        print("OK     ", text)
    else:
        print("FALSCH ", text, "-> bekommen", repr(ist), ", erwartet", repr(soll))


def pruefe_nah(ist, soll, text, tol=1e-6):
    """Vergleicht Kommazahlen mit Toleranz."""
    if math.isclose(ist, soll, rel_tol=tol, abs_tol=tol):
        print("OK     ", text)
    else:
        print("FALSCH ", text, "-> bekommen", ist, ", erwartet", soll)


def mutanten_pruefen(test, richtig, varianten, text="Tests"):
    """Lässt die Testfunktion test(f) gegen die richtige Funktion und gegen fehlerhafte Varianten laufen.
    Gute Tests bestehen bei der richtigen Funktion und scheitern bei JEDER fehlerhaften Variante.
    varianten: Liste von (Beschreibung, Funktion)."""
    try:
        test(richtig)
    except AssertionError as fehler:
        print("NICHT BESTANDEN:", text, "scheitern schon bei der richtigen Funktion:", fehler)
        return False
    except Exception as fehler:
        print("NICHT BESTANDEN:", text, "brechen mit", type(fehler).__name__, "ab:", fehler)
        return False
    entdeckt = 0
    for beschreibung, f in varianten:
        try:
            test(f)
        except Exception:
            entdeckt = entdeckt + 1
            print("  gefunden  :", beschreibung)
        else:
            print("  UNENTDECKT:", beschreibung)
    if entdeckt == len(varianten):
        print("BESTANDEN:", text, "finden alle", len(varianten), "fehlerhaften Varianten.")
        return True
    print("NICHT BESTANDEN:", text, "finden nur", entdeckt, "von", len(varianten), "Varianten. Welcher Testfall fehlt?")
    return False


print("Werkzeuge bereit: pruefe, pruefe_nah, mutanten_pruefen")

## Stufe 1 – Aufwärmen

### 1.1 Vorhersagen: assert schweigt
`assert bedingung, "Meldung"` tut nichts, wenn die Bedingung wahr ist, und löst sonst einen `AssertionError` aus. Welche Zeile bricht ab, und was steht im Fehler? Schreib die Vorhersage in den Kommentar, dann führe aus. Kommentiere die abbrechende Zeile danach aus, damit der Rest läuft.

In [ ]:
# Meine Vorhersage: ...
def kmh_in_ms(v):
    return v * 3.6          # Fehler: mal statt durch

assert kmh_in_ms(0) == 0.0
print("Test 1 bestanden")
assert kmh_in_ms(36) == 10.0, "36 km/h müssen 10 m/s sein"
print("Test 2 bestanden")

### 1.2 Vorhersagen: except AssertionError as fehler
Ein gescheiterter `assert` lässt sich mit `try`/`except` abfangen. Das kleine Wort `as` gibt der Ausnahme einen Namen. Was gibt das Programm aus? Wie oft läuft die Schleife durch?

In [ ]:
# Meine Vorhersage: ...
def zustand_ki(t):
    if t < 70:
        return "normal"
    elif t < 85:
        return "warnung"
    else:
        return "abschalten"

gescheitert = 0
for t, erwartet in [(20, "normal"), (70, "warnung"), (85, "warnung"), (90, "abschalten")]:
    try:
        assert zustand_ki(t) == erwartet, f"{t} °C: erwartet {erwartet}, bekommen {zustand_ki(t)}"
    except AssertionError as fehler:
        gescheitert = gescheitert + 1
        print("GESCHEITERT:", fehler)
print(gescheitert, "von 4 gescheitert")

### 1.3 Selbsttest: Testwerte aus der Spezifikation ableiten
Eine Funktion `ampel(n)` ordnet die Drehzahl n in 1/min ein: unter 0 ungültig, von 0 bis unter 3000 grün, von 3000 bis einschließlich 4500 gelb, darüber rot. Trage in `bereiche` je einen typischen Wert pro Bereich ein (vier Werte) und in `grenzen` jede Grenze von **beiden Seiten** (sechs Werte, bei ganzen Zahlen die Nachbarn). Keine Funktion, nur Zahlen: Du denkst in der Spezifikation.

In [ ]:
bereiche = []        # je Bereich ein typischer Wert, zum Beispiel [-500, ...]
grenzen = []         # jede Grenze von beiden Seiten, zum Beispiel [-1, 0, ...]

pruefe(len(bereiche), 4, "vier typische Werte")
pruefe(len(grenzen), 6, "sechs Grenzwerte")
fehlend = [w for w in [-1, 0, 2999, 3000, 4500, 4501] if w not in grenzen]
pruefe(fehlend, [], "die Grenzwerte liegen richtig (es fehlen: " + str(fehlend) + ")")
pruefe(len([w for w in bereiche if w < 0]), 1, "ein Wert im ungültigen Bereich")
pruefe(len([w for w in bereiche if w >= 0 and w < 3000]), 1, "ein Wert im grünen Bereich")
pruefe(len([w for w in bereiche if w >= 3000 and w <= 4500]), 1, "ein Wert im gelben Bereich")
pruefe(len([w for w in bereiche if w > 4500]), 1, "ein Wert im roten Bereich")

### 1.4 Selbsttest: die erste Testfunktion
Schreibe `test_kmh(f)` mit drei `assert`-Zeilen für die Funktion `f`, die km/h in m/s umrechnet (durch 3,6): Null, 36 km/h gleich 10 m/s und −72 km/h gleich −20 m/s. Die Testfunktion bekommt die zu prüfende Funktion als Parameter `f`, damit der Selbsttest sie gegen fehlerhafte Varianten laufen lassen kann. Vergleiche Kommazahlen mit `math.isclose`.

In [ ]:
def kmh_richtig(v):
    return v / 3.6

def kmh_mal(v):
    return v * 3.6

def kmh_betrag(v):
    return abs(v) / 3.6

def test_kmh(f):
    # Dein Code hier: drei assert-Zeilen mit f(...) und math.isclose
    pass

mutanten_pruefen(test_kmh, kmh_richtig, [("mal 3,6 statt durch 3,6", kmh_mal), ("Betrag statt Vorzeichen", kmh_betrag)], "Deine Tests")

## Stufe 2 – Anwenden

### 2.1 Tests für das Briefporto
Die Funktion `porto(gramm)`: bis einschließlich 20 g 0,95 €, bis einschließlich 50 g 1,60 €, bis einschließlich 500 g 2,75 €, sonst (0 oder weniger, über 500 g) `None`. Schreibe `test_porto(f)` mit mindestens zwölf `assert`-Zeilen oder mit einer Liste von Tupeln und Schleife: je Bereich ein typischer Wert, jede Grenze von beiden Seiten, Sonderfälle. Der Selbsttest lässt deine Tests gegen sechs fehlerhafte Varianten laufen. Für `None` vergleichst du mit `is None`.

In [ ]:
def porto_richtig(gramm):
    if gramm <= 0 or gramm > 500:
        return None
    elif gramm <= 20:
        return 0.95
    elif gramm <= 50:
        return 1.60
    return 2.75

def porto_null(gramm):              # 0 g wird berechnet
    if gramm < 0 or gramm > 500:
        return None
    elif gramm <= 20:
        return 0.95
    elif gramm <= 50:
        return 1.60
    return 2.75

def porto_20(gramm):                # Grenze 20 g: < statt <=
    if gramm <= 0 or gramm > 500:
        return None
    elif gramm < 20:
        return 0.95
    elif gramm <= 50:
        return 1.60
    return 2.75

def porto_50(gramm):                # Grenze 50 g: < statt <=
    if gramm <= 0 or gramm > 500:
        return None
    elif gramm <= 20:
        return 0.95
    elif gramm < 50:
        return 1.60
    return 2.75

def porto_500(gramm):               # Grenze 500 g: 500 g nicht mehr versendbar
    if gramm <= 0 or gramm >= 500:
        return None
    elif gramm <= 20:
        return 0.95
    elif gramm <= 50:
        return 1.60
    return 2.75

def porto_ohne_limit(gramm):        # keine Obergrenze
    if gramm <= 0:
        return None
    elif gramm <= 20:
        return 0.95
    elif gramm <= 50:
        return 1.60
    return 2.75

def porto_dreher(gramm):            # Zahlendreher im Preis
    if gramm <= 0 or gramm > 500:
        return None
    elif gramm <= 20:
        return 0.95
    elif gramm <= 50:
        return 1.60
    return 2.57

porto_varianten = [
    ("0 g wird berechnet", porto_null),
    ("Grenze 20 g: < statt <=", porto_20),
    ("Grenze 50 g: < statt <=", porto_50),
    ("Grenze 500 g: 500 g nicht mehr versendbar", porto_500),
    ("keine Obergrenze", porto_ohne_limit),
    ("Zahlendreher im Preis", porto_dreher),
]

def test_porto(f):
    # Dein Code hier
    pass

mutanten_pruefen(test_porto, porto_richtig, porto_varianten, "Deine Porto-Tests")

### 2.2 Traceback über mehrere Funktionen lesen
Führe die Zelle aus. Sie fängt den Fehler ab und druckt den Traceback aus, damit das Notebook weiterläuft. Lies **von unten nach oben**: Fehlerart und Meldung (was?), Fehlerstelle (wo?), Aufrufer (wie kam es dazu?). Beantworte in den Kommentaren, dann repariere `lese_wert` so, dass `mittelwert(daten)` 63.333... liefert, und prüfe mit dem Selbsttest.

In [ ]:
# Fehlerart: ...
# Fehlerstelle (Funktion, Zeile): ...
# Ursache: ...

def lese_wert(text):
    return float(text)

def mittelwert(texte):
    summe = 0.0
    for text in texte:
        summe = summe + lese_wert(text)
    return summe / len(texte)

daten = ["62,5", "64,0", "63,5"]
try:
    print(mittelwert(daten))
except Exception:
    traceback.print_exc()

try:
    ergebnis = mittelwert(daten)
except ValueError:
    ergebnis = None
pruefe(ergebnis is not None and math.isclose(ergebnis, 63.333333, abs_tol=1e-4), True, "mittelwert(daten) liefert 63.333...")

### 2.3 Fehler finden: Hypothese, Beobachten, Korrigieren
`staerkste_bremsung(werte)` soll den kleinsten Wert einer Beschleunigungsreihe liefern. Beim Anfahren ohne Bremsung gibt sie 0.0 statt 0.2 aus. Kein Traceback: ein Logikfehler. Gehe den Zyklus durch: (1) Hypothese in den Kommentar, (2) Beobachten mit `print` an der verdächtigen Stelle **oder** mit dem Debugger (Haltepunkt in der Schleife, Variable `kleinster` im Variablenfenster), (3) eine Änderung, (4) Selbsttest erneut.

In [ ]:
# Meine Hypothese: ...

def staerkste_bremsung(werte):
    kleinster = 0.0
    for a in werte:
        if a < kleinster:
            kleinster = a
    return kleinster

pruefe(staerkste_bremsung([0.4, 0.2, 0.9, 0.5]), 0.2, "Anfahren ohne Bremsung")
pruefe(staerkste_bremsung([0.5, -1.5, -3.8, -2.0]), -3.8, "Bremsung")
pruefe(staerkste_bremsung([-0.5, -2.5, -1.5]), -2.5, "nur negative Werte")
pruefe(staerkste_bremsung([5.0]), 5.0, "ein einziger Wert")

### 2.4 Selbsttest: Tests für eine Liste
Die Funktion `anzahl_ueber(werte, grenze)` zählt die Werte, die **größer als** die Grenze sind (ein Wert genau auf der Grenze zählt nicht). Eine leere Liste gibt 0. Schreibe `test_anzahl(f)` mit mindestens sieben Fällen als Liste von Tupeln `((werte, grenze), erwartet)` und einer Schleife. Denke an: leere Liste, ein Wert genau auf der Grenze, alle darüber, keiner darüber, negative Zahlen.

In [ ]:
def anzahl_richtig(werte, grenze):
    n = 0
    for w in werte:
        if w > grenze:
            n = n + 1
    return n

def anzahl_gleich(werte, grenze):
    n = 0
    for w in werte:
        if w >= grenze:
            n = n + 1
    return n

def anzahl_erstes_fehlt(werte, grenze):
    n = 0
    for w in werte[1:]:
        if w > grenze:
            n = n + 1
    return n

def anzahl_immer_eins(werte, grenze):
    for w in werte:
        if w > grenze:
            return 1
    return 0

def test_anzahl(f):
    faelle = [
        # (([...], grenze), erwartet),
    ]
    for (werte, grenze), erwartet in faelle:
        assert f(werte, grenze) == erwartet, f"f({werte}, {grenze}) soll {erwartet} sein"
    assert len(faelle) >= 7, "Es fehlen Testfälle (mindestens sieben)"

mutanten_pruefen(test_anzahl, anzahl_richtig, [("Wert auf der Grenze zählt mit (>= statt >)", anzahl_gleich), ("das erste Element wird übersprungen", anzahl_erstes_fehlt), ("zählt höchstens 1", anzahl_immer_eins)], "Deine Zähl-Tests")

## Stufe 3 – Knobeln

### 3.1 Schutzfunktion: höchstens sechs Testfälle
Eine Maschine darf nur starten, wenn die Tür geschlossen ist **und** der Not-Halt nicht gedrückt ist **und** (die Spindel steht **oder** der Anlauf quittiert wurde). Vier Wahrheitswerte ergeben 16 Kombinationen: Du darfst nur **höchstens sechs** testen, und trotzdem muss jede fehlerhafte Variante scheitern. Überlege, welche Kombination jede Bedingung einzeln „beweist“. Fünf reichen.

In [ ]:
def schutz_richtig(tuer_zu, notaus, steht, quittiert):
    return tuer_zu and not notaus and (steht or quittiert)

def schutz_und(tuer_zu, notaus, steht, quittiert):          # Spindel UND quittiert
    return tuer_zu and not notaus and (steht and quittiert)

def schutz_nur_spindel(tuer_zu, notaus, steht, quittiert):  # nur die Spindel zählt
    return tuer_zu and not notaus and steht

def schutz_nur_quittiert(tuer_zu, notaus, steht, quittiert):  # nur die Quittierung zählt
    return tuer_zu and not notaus and quittiert

def schutz_notaus(tuer_zu, notaus, steht, quittiert):       # das not vor dem Not-Halt fehlt
    return tuer_zu and notaus and (steht or quittiert)

def schutz_ohne_tuer(tuer_zu, notaus, steht, quittiert):    # die Schutztür wird nicht geprüft
    return (not notaus) and (steht or quittiert)

def schutz_ohne_anlauf(tuer_zu, notaus, steht, quittiert):  # Spindel und Quittierung werden nicht geprüft
    return tuer_zu and not notaus

def schutz_oder(tuer_zu, notaus, steht, quittiert):         # Tür oder kein Not-Halt statt und
    return (tuer_zu or not notaus) and (steht or quittiert)

schutz_varianten = [
    ("Spindel UND quittiert", schutz_und),
    ("nur die Spindel zählt", schutz_nur_spindel),
    ("nur die Quittierung zählt", schutz_nur_quittiert),
    ("das not vor dem Not-Halt fehlt", schutz_notaus),
    ("die Schutztür wird nicht geprüft", schutz_ohne_tuer),
    ("Spindel und Quittierung werden nicht geprüft", schutz_ohne_anlauf),
    ("Tür oder kein Not-Halt statt und", schutz_oder),
]

def test_schutz(f):
    faelle = [
        # (tuer_zu, notaus, steht, quittiert, erwartet),
    ]
    assert len(faelle) >= 1 and len(faelle) <= 6, "höchstens sechs Fälle"
    for tuer_zu, notaus, steht, quittiert, erwartet in faelle:
        assert f(tuer_zu, notaus, steht, quittiert) == erwartet, f"Fall {(tuer_zu, notaus, steht, quittiert)}: erwartet {erwartet}"

mutanten_pruefen(test_schutz, schutz_richtig, schutz_varianten, "Deine Schutz-Tests")

### 3.2 Eigene Mutanten bauen
Jetzt drehst du die Rolle um: Du baust die fehlerhaften Varianten. Schreibe für `gebuehr(gewicht)` (bis 5 kg 4,90 €, bis 10 kg 6,90 €, bis 31,5 kg 12,90 €, sonst `None`) **drei fehlerhafte Varianten** mit je einem eingebauten Fehler an einer Grenze. Der Selbsttest prüft, dass jede Variante von der richtigen Funktion abweicht und dass sie *zu den Grenzen gehört* (sie weicht bei einem Grenzwert ab, nicht bei jedem Wert). Das ist die Technik hinter Mutationstests.

In [ ]:
def gebuehr(gewicht):
    if gewicht <= 0 or gewicht > 31.5:
        return None
    elif gewicht <= 5:
        return 4.90
    elif gewicht <= 10:
        return 6.90
    return 12.90

def variante_1(gewicht):
    # Dein Code hier: Kopie von gebuehr mit einem Fehler an einer Grenze
    return gebuehr(gewicht)

def variante_2(gewicht):
    return gebuehr(gewicht)

def variante_3(gewicht):
    return gebuehr(gewicht)

grenzwerte = [0, 5, 5.1, 10, 10.1, 31.5, 31.6]
nr = 0
ergebnisse = []
for v in [variante_1, variante_2, variante_3]:
    nr = nr + 1
    abweichungen = [w for w in grenzwerte if v(w) != gebuehr(w)]
    allgemein = [w for w in [2, 8, 20] if v(w) != gebuehr(w)]
    pruefe(len(abweichungen) >= 1 and len(allgemein) == 0, True, f"variante_{nr}: weicht nur an einer Grenze ab (Abweichungen bei {abweichungen})")
    ergebnisse.append([v(w) for w in grenzwerte])
verschieden = ergebnisse[0] != ergebnisse[1] and ergebnisse[0] != ergebnisse[2] and ergebnisse[1] != ergebnisse[2]
pruefe(verschieden, True, "die drei Varianten sind verschieden")

### 3.3 Die KI schreibt, du prüfst: Tests vor dem Code
Das ist der Arbeitsablauf aus Phase 2, und diese Aufgabe führst du **mit Copilot** durch (in Codespaces):

1. Wähle eine Spezifikation: zum Beispiel `bewerte_vibration(v_eff)` (Schwingstärke in mm/s, hier frei festgelegt und keine Norm: bis einschließlich 2,8 „gut“, bis einschließlich 7,1 „zulässig“, bis einschließlich 18 „bedingt zulässig“, darüber „nicht zulässig“; negative Werte „ungültig“).
2. Schreibe **zuerst** deine Testfälle von Hand in die Zelle unten, mit Bereichen und Grenzen.
3. Bitte Copilot, die Funktion zu schreiben, und füge sie in die Zelle ein.
4. Lass deine Tests laufen. Scheitert einer, ist das ein Fund: Hypothese, Beobachten, Korrigieren (von Hand oder mit Copilot als Erklärer).
5. Bitte Copilot danach um **seine** Testfälle. Vergleiche: Welche Grenzen hat er vergessen?

Trage unten deine Ergebnisse ein.

In [ ]:
# 1. Meine Spezifikation (Bereiche und Grenzen):
# 2. Meine Testfälle von Hand (vor dem Code der KI):
faelle = [
    # (v_eff, erwartetes Urteil),
]

# 3. Die Funktion der KI (hier einfügen):
def bewerte_vibration(v_eff):
    return None

# 4. Meine Tests gegen die Funktion der KI:
gescheitert = 0
for v, erwartet in faelle:
    try:
        assert bewerte_vibration(v) == erwartet, f"{v} mm/s: erwartet {erwartet}, bekommen {bewerte_vibration(v)}"
    except AssertionError as fehler:
        gescheitert = gescheitert + 1
        print("GESCHEITERT:", fehler)
print(gescheitert, "von", len(faelle), "gescheitert")

# 5. Notiere hier: Welche Grenzen hat die KI in ihren eigenen Tests vergessen?

## Anhang – Ausblick: pytest (freiwillig)

`pytest` ist das Test-Gerüst der Praxis (Ampel: Delegieren). Es findet alle Funktionen mit `test_` am Anfang in Dateien `test_*.py` selbst, führt sie aus und zeigt bei einem Fehler genau, welcher Wert erwartet und welcher bekommen wurde. Es läuft in Codespaces oder lokal nach `pip install pytest`, **nicht im Browser**. Die folgende Zelle schreibt zwei Dateien und startet pytest. Führe sie nur aus, wenn `pytest` installiert ist.

In [ ]:
%%writefile lager.py
def zustand(t):
    if t < -40 or t > 200:
        return "sensorfehler"
    if t < 70:
        return "normal"
    elif t < 85:          # der KI-Fehler: bei 85 soll noch Warnung gelten (< statt <=)
        return "warnung"
    else:
        return "abschalten"

In [ ]:
%%writefile test_lager.py
from lager import zustand

def test_grenze_85():
    assert zustand(85) == "warnung", "Grenzwert 85 °C falsch eingeordnet"

def test_faelle():
    faelle = [(20, "normal"), (69.9, "normal"), (70, "warnung"), (85.1, "abschalten"), (-40.1, "sensorfehler")]
    for t, erwartet in faelle:
        assert zustand(t) == erwartet

In [ ]:
# Im Terminal: pytest   (hier in der Zelle:)
!python -m pytest -q test_lager.py

Du siehst: `test_faelle` besteht, `test_grenze_85` scheitert, und pytest zeigt dir die Zeile und die beiden Werte. Ändere in `lager.py` die Grenze auf `<=` und führe die Zelle noch einmal aus. Das ist der gesamte Ablauf: Test schreiben, scheitern sehen, korrigieren, Test erneut.